In [1]:
#!pip install GitPython

In [1]:
import os

api_key = os.getenv("AICREDITS_API_KEY")
base_url = os.getenv("AICREDITS_BASE_URL")
os.environ["GIT_PYTHON_GIT_EXECUTABLE"] = r"C:\Program Files\Git\cmd\git.exe"

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import GitLoader

repo_loader = GitLoader(
    repo_path="./genai_repo",
    clone_url="https://github.com/ShivaUlavi/GenAI",
    branch="main",
    file_filter=lambda file_path: file_path.endswith(".ipynb")
)

documents = repo_loader.load()
print(f"Successfully loaded {len(documents)} document(s)!")

C:\Users\Shivachetan Ulavi\AppData\Local\Temp\ipykernel_11584\3677836528.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import GitLoader


Successfully loaded 29 document(s)!


In [4]:
#documents

In [5]:
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter

python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=50, chunk_overlap=0
)

# Split the Document into chunks for embedding and vector storage
texts = python_splitter.split_documents(documents)   

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

embeddings = OpenAIEmbeddings(api_key=api_key, base_url=base_url)
llm = ChatOpenAI(model = "meta-llama/llama-3.1-70b-instruct",
                api_key= api_key,
                base_url=base_url)

db = Chroma.from_documents(texts, embeddings)
retriever = db.as_retriever(search_type="mmr", search_kwargs={"k":8})

# Create a retrieval chain for Q&A over code
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the user's questions based on the below context:\n\n{context}"),
    ("placeholder", "{chat_history}"),
    ("user", "{input}"),
])

document_chain = create_stuff_documents_chain(llm, prompt)
qa = create_retrieval_chain(retriever, document_chain)


In [14]:
# Ask a question from the book
question = "How does the reflection loop or feedback mechanism work in the multi-agent notebook?"
result = qa.invoke({"input": question})
print(result["answer"])

In the multi-layered simulation, the reflection loop or feedback mechanism works as follows:

1. **Observation**: The agent observes the outcomes of its actions in the simulation.
2. **Reflection**: The agent reflects on these outcomes, using its tools and planning capabilities to analyze what happened.
3. **Feedback loops**: The reflection process generates feedback, which causes localized "feedback loops" in the simulation. This means that the agent's reflections and adjustments affect the simulation, which in turn affects the agent's next observations.

This continuous cycle of observation, reflection, and feedback enables the agent to learn, adapt, and improve its performance within the multi-layered simulation.


In [15]:
# Defining a list of questions tailored to my repo
questions = [
    "How does the reflection loop or feedback mechanism work in the multi-agent notebook?",
    "What are the key nodes and edges defined in the LangGraph multi-agent architecture?",
    "How is state passed between the different agents or steps in the workflow?"
]

# Iterate through each question and streaming the answers
for index, question in enumerate(questions, 1):
    print(f"\n[Question {index}]: {question}")
    print("-" * 60)
    print("Answer: ", end="", flush=True)
    
    # Stream response chunks from the retrieval chain
    for chunk in qa.stream({"input": question}):
        if "answer" in chunk:
            print(chunk["answer"], end="", flush=True)
            
    print("\n" + "=" * 60)


[Question 1]: How does the reflection loop or feedback mechanism work in the multi-agent notebook?
------------------------------------------------------------
Answer: In the multi-layered simulation, the reflection loop or feedback mechanism works as follows:

1. **Observation**: The agent observes the outcomes of its actions in the simulation.
2. **Reflection**: The agent reflects on these outcomes, using its tools and thought processes to analyze and interpret the results.
3. **Feedback loops**: The reflection process causes localized "feedback loops" in the simulation, where the agent's thoughts and actions influence the simulation, which in turn affects the agent's observations, and so on.

This creates a continuous cycle of observation, reflection, and feedback, allowing the agent to learn and adapt within the multi-layered simulation.

[Question 2]: What are the key nodes and edges defined in the LangGraph multi-agent architecture?
----------------------------------------------

### Code Generation

In [16]:
question = "Write a Python script demonstrating a simple Chain of Thought reasoning loop for an agent using LangChain."

print("Generating CoT Workflow...\n")
for chunk in qa.stream({"input": question}):
    if "answer" in chunk:
        print(chunk["answer"], end="", flush=True)

Generating CoT Workflow...

Here's a simple example of a Chain of Thought reasoning loop for an agent using LangChain:

```python
from langchain import LLMChain, PromptTemplate
from langchain.agents import Agent
from langchain.llms import HuggingFaceHubLLM

# Initialize the LLM
llm = HuggingFaceHubLLM(model_name="llama-2b-chinese")

# Define a prompt template for the reasoning loop
template = PromptTemplate(
    input_variables=["question"],
    template="Let's think step by step to answer this question: {question}. Step 1: Think about the overall goal of the problem.",
)

# Define a Chain of Thought approach for the agent
chain = LLMChain(llm=llm, prompt=template)

# Define an agent that uses the Chain of Thought approach
agent = Agent(chain=chain, verbose=True)

# Test the agent with a question
question = "What is the capital of France?"
agent.run(question)
```

In this example, we:

1. Initialize a large language model (LLM) using the Hugging Face Hub.
2. Define a prompt template fo

In [18]:
question = (
    "Can you write a simple LangGraph workflow implementation in Python "
    "using StateGraph, defining a couple of nodes and edges?"
)

print("Generating LangGraph Workflow...\n")
for chunk in qa.stream({"input": question}):
    if "answer" in chunk:
        print(chunk["answer"], end="", flush=True)

Generating LangGraph Workflow...

Below is an example of a simple LangGraph workflow implementation using StateGraph in Python:

```python
from langgraph.graph import StateGraph, Node, Edge

# Create a new StateGraph
graph = StateGraph()

# Define nodes
node1 = Node("start", "This is the starting node")
node2 = Node("process", "This is a processing node")
node3 = Node("end", "This is the ending node")

# Add nodes to the graph
graph.add_node(node1)
graph.add_node(node2)
graph.add_node(node3)

# Define edges
edge1 = Edge(node1, node2, "This is the edge from start to process")
edge2 = Edge(node2, node3, "This is the edge from process to end")

# Add edges to the graph
graph.add_edge(edge1)
graph.add_edge(edge2)

# Print the graph
print(graph)

# Run the graph
graph.run()
```

This implementation defines a StateGraph with three nodes: `start`, `process`, and `end`. It then adds two edges connecting the nodes: one from `start` to `process` and another from `process` to `end`. Finally, it p

### The model hallucinated a bit here by inventing Node() and Edge() classes and calling graph.run(). In actual LangGraph, nodes are just regular Python functions, you pass them in via add_node("name", function_name), and you execute the graph by calling .compile() and .invoke().

In [21]:
question = (
    "Can you write a simple LangGraph workflow implementation in Python "
    "using StateGraph, defining a couple of nodes and edges?"
)

print("Generating LangGraph Workflow...\n")
generated_output = "" # Initialize an empty string to accumulate the stream

# 1. Stream the response and capture it live
for chunk in qa.stream({"input": question}):
    if "answer" in chunk:
        token = chunk["answer"]
        print(token, end="", flush=True)
        generated_output += token  # Append each token to our string variable

print("\n\n" + "=" * 60)
print("Running Validator LLM on Captured Output...\n")


##Validation of outputs
from langchain_core.prompts import PromptTemplate

# Define the validation template
validation_template = """Analyze the following Python code for:
1. Potential security vulnerabilities
2. Logic errors or framework hallucinations (e.g., non-existent classes, incorrect API usage)
3. Performance issues
4. Edge case handling

Code to analyze:
```python
{generated_code}
Provide a detailed analysis with specific issues and recommended corrected code.
"""

validation_prompt = PromptTemplate(
input_variables=["generated_code"],
template=validation_template
)


llm_val = ChatOpenAI(model = "anthropic/claude-sonnet-4.6", api_key=api_key, base_url=base_url)
validation_chain = validation_prompt | llm_val

# 2. Pass the dynamically captured output straight into your validator
validation_response = validation_chain.invoke({"generated_code": generated_output})
print(validation_response.content)

Generating LangGraph Workflow...

Here is a simple example of a LangGraph workflow implementation in Python using `StateGraph`:
```python
from langgraph.graph import StateGraph

# Create a new StateGraph
graph = StateGraph()

# Define nodes
node1 = graph.add_node("Node 1", state="initial")
node2 = graph.add_node("Node 2", state="pending")
node3 = graph.add_node("Node 3", state="completed")

# Define edges
graph.add_edge(node1, node2, transition="start")
graph.add_edge(node2, node3, transition="finish")

# Print the graph
print(graph)
```
This code creates a `StateGraph` instance and defines three nodes: "Node 1", "Node 2", and "Node 3". It then defines two edges: one from "Node 1" to "Node 2" with a transition label "start", and another from "Node 2" to "Node 3" with a transition label "finish".

You can customize the node and edge definitions to fit your specific use case.

Note that the `StateGraph` class is not a standard Python library, so you'll need to have the `langgraph` librar